# Workspace Setup Notebook
This notebook continues the new workspace setup in VS Code and validates a clean Python environment for this project.

## 1) Create Project Folder Structure
Create a standard project layout and verify it programmatically.

In [ ]:
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
print(f'Project root: {project_root}')

for folder in ['src', 'notebooks', 'data/raw', 'data/processed', 'tests', '.vscode']:
    (project_root / folder).mkdir(parents=True, exist_ok=True)

structure = sorted(str(p.relative_to(project_root)) for p in project_root.iterdir())
print('Top-level entries:')
for entry in structure:
    print('-', entry)

## 2) Initialize and Activate a Virtual Environment
Create a local `.venv`, inspect interpreter paths, and confirm activation expectations.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
venv_dir = project_root / '.venv'

if not venv_dir.exists():
    subprocess.run([sys.executable, '-m', 'venv', str(venv_dir)], check=True)
    print('Created .venv')
else:
    print('.venv already exists')

venv_python = venv_dir / 'Scripts' / 'python.exe'
print('Current interpreter :', sys.executable)
print('Workspace venv python:', venv_python)
print('VIRTUAL_ENV env var  :', os.environ.get('VIRTUAL_ENV', '<not set>'))

## 3) Install Core Packages
Install core dependencies and generate a reproducible requirements snapshot.

In [ ]:
import subprocess
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
venv_python = project_root / '.venv' / 'Scripts' / 'python.exe'

packages = ['jupyter', 'ipykernel', 'numpy', 'pandas', 'matplotlib', 'seaborn', 'requests', 'pytest']
subprocess.run([str(venv_python), '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([str(venv_python), '-m', 'pip', 'install', *packages], check=True)

req_file = project_root / 'requirements.lock.txt'
with req_file.open('w', encoding='utf-8') as f:
    freeze = subprocess.run([str(venv_python), '-m', 'pip', 'freeze'], check=True, capture_output=True, text=True)
    f.write(freeze.stdout)

print(f'Wrote reproducible snapshot: {req_file}')

## 4) Register the Workspace Jupyter Kernel
Register an ipykernel for this workspace virtual environment and verify that it appears in the kernel list.

In [ ]:
import subprocess
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
venv_python = project_root / '.venv' / 'Scripts' / 'python.exe'
kernel_name = 'cryptoan-venv'

subprocess.run([
    str(venv_python),
    '-m',
    'ipykernel',
    'install',
    '--user',
    '--name',
    kernel_name,
    '--display-name',
    'Python (.venv) - cryptoan',
], check=True)

kernels = subprocess.run([str(venv_python), '-m', 'jupyter', 'kernelspec', 'list'], check=True, capture_output=True, text=True)
print(kernels.stdout)

## 5) Generate VS Code Workspace Settings
Create `.vscode/settings.json` so VS Code defaults to the workspace `.venv` interpreter and notebook-friendly behavior.

In [ ]:
import json
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
vscode_dir = project_root / '.vscode'
vscode_dir.mkdir(parents=True, exist_ok=True)

settings = {
    'python.defaultInterpreterPath': '${workspaceFolder}\\.venv\\Scripts\\python.exe',
    'python.testing.pytestEnabled': True,
    'python.testing.pytestArgs': ['tests'],
    'jupyter.askForKernelRestart': False,
    'jupyter.notebookFileRoot': '${workspaceFolder}',
}

settings_path = vscode_dir / 'settings.json'
with settings_path.open('w', encoding='utf-8') as f:
    json.dump(settings, f, indent=2)

print(f'Created: {settings_path}')

## 6) Add a Smoke Test Script and Run It
Create a script that verifies imports, interpreter details, and a simple numeric computation.

In [ ]:
import subprocess
import textwrap
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
venv_python = project_root / '.venv' / 'Scripts' / 'python.exe'
smoke_path = project_root / 'tests' / 'smoke_test.py'

smoke_code = textwrap.dedent('''
    import platform
    import sys

    import matplotlib
    import numpy as np
    import pandas as pd

    print('Interpreter:', sys.executable)
    print('Python     :', platform.python_version())
    print('numpy      :', np.__version__)
    print('pandas     :', pd.__version__)
    print('matplotlib :', matplotlib.__version__)

    arr = np.array([1, 2, 3, 4, 5])
    print('Mean value :', arr.mean())
''').strip() + '\n'

smoke_path.write_text(smoke_code, encoding='utf-8')
print(f'Created: {smoke_path}')

result = subprocess.run([str(venv_python), str(smoke_path)], check=True, capture_output=True, text=True)
print(result.stdout)